In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from smlm.io import load_raw
from smlm import viz

src = r"G:\SPARK\Cy5\20261004\SPARK Cy5-SO3-antibody-638nm-16mW-0p1s-3000frames-fov14_raw.tif"
raw = load_raw(src)
print(raw.data.shape, raw.em_gain)   # em_gain 为 None
pixel_size=106.6 # nm/pixel

In [ ]:
# 接下来交给 camera 层转光子
from smlm.camera import EMCCD
cam = EMCCD(conv_factor=5.75, em_gain=300)
image = cam.process(raw, pixel_size=106.6)

In [ ]:
from smlm.imgprocess import Pipeline, Crop, SelectFrames
prep = Pipeline(Crop(y=(0,256), x=(0, 256)),
                SelectFrames(stop=None))
stack = prep(image)
print(stack.shape)

## Load Locs

In [ ]:
# ===== 选择要分析的数据集（DECODE 定位 + 对应原始帧）=====
import numpy as np
# 每次 DECODE 推理的结果都写在各自的目录里，不会互相覆盖：
#   E:\CodingProj\DECODE-master\_real_fit_preview\<dataset>\<model_tag>\real_locs_all.npz
DATASET = 'fov14'      # 'fov14' | 'tubulin_fov8' | 'mitro_fov4' | 'lyso_fov10roi2'

DATASETS = {
    'fov14': dict(
        locs=r"E:\CodingProj\DECODE-master\_real_fit_preview\fov14_cy5so3\sigma16_raw_256x256\real_locs_all.npz",
        tif=r"G:\SPARK\Cy5\20261004\SPARK Cy5-SO3-antibody-638nm-16mW-0p1s-3000frames-fov14_raw.tif",
        origin=(0.0, 0.0),      # DECODE 处理后帧的 (x, y) 原点在原始 TIF 中的位置
        px_size=106.6,          # nm / pixel
        ex_time=0.025,          # s
        units='ADU (camera.backward)',
    ),
    'tubulin_fov8': dict(
        locs=r"E:\CodingProj\DECODE-master\_real_fit_preview\tubulin_fov8\sigma16_512x512\real_locs_all.npz",
        tif=r"G:\SPARK\SPARK\Tubulin\SPARK Cy5-SO3H-200x_dish1_fov8_Tifstack_Uint16.tif",
        origin=(0.0, 0.0),      # 512 能被 8 整除 -> AutoCenterCrop(8) 不裁切
        px_size=106.6,
        ex_time=0.025,
        units='photons (already converted, no backward)',
    ),
    'mitro_fov4': dict(
        locs=r"E:\CodingProj\DECODE-master\_real_fit_preview\mitro_fov4\sigma16_512x512\real_locs_all.npz",
        tif=r"G:\SPARK\SPARK\Mitro\Cy5pH-TPP-10uM_638nm_50frames_0p05s_10mW_fov4.tif",
        origin=(0.0, 0.0),
        px_size=106.6,
        ex_time=0.05,
        units='ADU (camera.backward)',
    ),
    'lyso_fov10roi2': dict(
        locs=r"E:\CodingProj\DECODE-master\_real_fit_preview\lyso_fov10roi2\sigma16_105x63\real_locs_all.npz",
        tif=r"G:\SPARK\SPARK\Lyso\Cy7_10uM_721nm_5000frames_0p1s_80mW_fov10ROI2.tif",
        origin=(1.0, 4.0),      # 105x63 -> AutoCenterCrop(8) -> 104x56, 原点 (x,y)=(1,4)
        px_size=106.6,
        ex_time=0.1,
        units='ADU (camera.backward)',
    ),
}
DS = DATASETS[DATASET]

# 用该数据集的原始帧重建 stack（DECODE 的 x/y 就在这个帧坐标系里）
import tifffile
from smlm.datatypes import ImageStack

_frames = tifffile.imread(DS['tif'])
stack = ImageStack(data=np.asarray(_frames, dtype=np.float32),
                   pixel_size=DS['px_size'],
                   metadata={'crop_offset': (0, 0), 'dataset': DATASET})
src = DS['tif']
pixel_size = DS['px_size']
if 'raw' in globals():
    raw.exposure_time = DS['ex_time']

print(f"dataset : {DATASET}   [{DS['units']}]")
print(f"  tif   : {DS['tif']}")
print(f"  stack : {stack.shape}  pixel_size={pixel_size} nm  ex_time={DS['ex_time']} s")
print(f"  locs  : {DS['locs']}")

In [ ]:
# ===== DECODE 定位结果 -> smlm Localizations =====
# 数据来源由上一格的 DATASET / DS 决定（默认 tubulin_fov8）。
# DECODE 输出的 (x, y) 是在 AutoCenterCrop(8) 之后的帧坐标系里，
# 这里按 DS['origin'] 与当前 stack 的 crop_offset 映射到 stack 坐标系。
from smlm.datatypes import Localizations
from smlm.viz import plot_distributions

_z = np.load(DS['locs'], allow_pickle=False)

if "stack" in globals():
    _y0, _x0 = stack.metadata.get("crop_offset", (0, 0))
else:
    _y0, _x0 = (0, 0)
_off_x = float(DS['origin'][0]) - int(_x0)
_off_y = float(DS['origin'][1]) - int(_y0)
print(f"coordinate offset applied: x{_off_x:+.0f}, y{_off_y:+.0f}")

locs = Localizations(
    frame=_z["frame"].astype(np.int64),
    x=(_z["x"] + _off_x).astype(np.float64),
    y=(_z["y"] + _off_y).astype(np.float64),
    photons=_z["photons"].astype(np.float64),
    bg=_z["bg"].astype(np.float64),
    sigma=_z["sigma"].astype(np.float64),   # PSF width in px (simulated Gaussian sigma)
    lp=_z["lp"].astype(np.float64),         # lateral localization precision in px
    z=None,
    z_lp=None,
    extra={
        "prob": _z["prob"].astype(np.float64),
        "sigma_x": _z["sigma_x"].astype(np.float64),
        "sigma_y": _z["sigma_y"].astype(np.float64),
    },
)

print(f"dataset: {DATASET}")
print(f"locs: {len(locs)} localizations")
print(f"frame range: {locs.frame.min()} .. {locs.frame.max()}")
print(f"x range (px): {locs.x.min():.2f} .. {locs.x.max():.2f}")
print(f"y range (px): {locs.y.min():.2f} .. {locs.y.max():.2f}")
print(f"photons median: {np.median(locs.photons):.1f}")
print(f"bg median: {np.median(locs.bg):.2f}")
print(f"lp median (px): {np.median(locs.lp):.4f}")
_nf = int(locs.frame.max()) + 1
_mid = _nf // 2
print(f"first half (0-{_mid - 1}): {(locs.frame < _mid).sum()} | "
      f"second half ({_mid}-{_nf - 1}): {(locs.frame >= _mid).sum()}")

plot_distributions(locs)

In [ ]:
ax = viz.overlay_points(stack, locs, frames=range(1,1000), color='r', marker='x', size=30, linewidth = 0.5, unit='pixel',pixel_size=pixel_size)

## Drift Corr

In [ ]:
import numpy as np
from smlm.datatypes import DriftTrace
from smlm.drift import CloudDriftEstimator
from smlm.imgprocess import DriftCorrector
from smlm.viz import plot_drift

# 只使用当前 stack 帧范围内的定位，避免帧数不一致
locs_drift = locs[locs.frame < stack.n_frames]

# ===== 开关：True=估计并校正漂移；False=零漂移（不校正，但仍做裁切）=====
USE_DRIFT = True

if USE_DRIFT:
    # 1) 估计(只需要定位)
    uncertainty = np.median(locs_drift.lp)
    est = CloudDriftEstimator(frames_per_seg=10, uncertainty=uncertainty, do_stage2=True)
    drift_trace = est.estimate(locs_drift, n_frames=stack.n_frames)
    print(f"drift estimated: total {drift_trace.total_drift:.2f} px over {drift_trace.n_frames} frames")
else:
    # 零漂移轨迹：dx = dy = 0，校正时图像与定位都不平移
    drift_trace = DriftTrace.zeros(stack.n_frames, pixel_size=pixel_size)
    print(f"USE_DRIFT=False -> zero drift trace ({drift_trace.n_frames} frames, total 0.00 px)")

# 2) 校正图像 + 定位
# 裁切窗口按视场大小自动计算，每边留 5 px 漂移缓冲
_h, _w = int(stack.shape[1]), int(stack.shape[2])
_crop = (max(8, _h - 10), max(8, _w - 10))
corr = DriftCorrector(crop=_crop, edge=5)
corr_image, corr_locs, keep_idx = corr.apply(trace=drift_trace, image=stack, locs=locs_drift)
print(f"field {_h}x{_w} -> corrected crop {corr_image.data.shape[1]}x{corr_image.data.shape[2]}, "
      f"locs kept {len(corr_locs)}/{len(locs_drift)}")

fig, ax = plot_drift(drift_trace, ex_time=raw.exposure_time,unit='nm',pixel_size=pixel_size) 


In [ ]:
ax = viz.overlay_points(corr_image, corr_locs, frames=np.arange(1,2000), color='r', marker='x', size=30, linewidth = 0.5)

## 重建

In [ ]:
from smlm.emitter import analyze_emitters_sm

# 首次出现帧窗口按当前数据集自适应（原来硬编码 (0, 9900) 只适用于 10000 帧的 fov14）
_ffr = (0, int(stack.n_frames) - 1)
print(f"first_frame_range = {_ffr}  (n_frames={stack.n_frames})")

emitters, emitters_raw = analyze_emitters_sm(
    corr_locs, n_frames=None,
    search_radius=0.6,
    max_gap=1000,
    min_frames=3,
    ecc_p=1e-3,
    first_frame_range=_ffr,
    motion='static', do_merge=False
)

In [ ]:
from smlm.emitter import collect_statistics
from smlm.viz import plot_trajectories, plot_bleaching_summary, plot_emitter_stats
stats = collect_statistics(emitters, locs, raw.exposure_time, 0)
plot_trajectories(corr_image.data, emitters, linewidth=0.7, frame_idx=np.arange(0, 2000), dpi=250)
plot_emitter_stats(stats)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

cyan = LinearSegmentedColormap.from_list('cyan', ['black', '#00FFFF'])


# 让图更容易看：
# 1) 左图保留“保守版有效不确定性”做颜色编码，适合看覆盖完整性和疑似问题 emitter
# 2) 右图改成“超分辨概率图”：把 emitter 作为带权重的 Gaussian 混合分布叠加成图像，
#    强调结构分布，不是看单个点；更适合作为可信的 SR 图像
# 3) 右图显式采用可信度加权，压低背景和高不确定性点，避免“噪声扩散”污染真正结构
# 4) 只保留 emitter centroid，避免重复闪烁叠加造成视觉模糊

render_factor = 8
# 可调参数：控制右图“背景 / 高不确定 emitter”的压低程度。
# 1) downweight_strength: 总压低强度，越大越压低
# 2) lp_cut_quantile: 超出该分位数的 emitter 被认定为高不确定，后续强烈压低
# 3) min_trust: 最低可信度门限，低于它的 emitter 直接丢掉
# 4) trusted_emitters_threshold: summary 输出里用到的可信 emitter 判断阈值
# 5) gmm_width_scale: 手动放大/缩小 GMM 的扩散宽度；如果 lp 偏小，可以调到 1.5~2.5
#
downweight_strength = 0
lp_cut_quantile = 100
min_trust = 1e-10
trusted_emitters_threshold = 0.15
gmm_width_scale = 1
n_emitters = len(emitters)
if n_emitters == 0:
    raise ValueError("No emitters available for SR rendering.")

h, w = corr_image.data.shape[1], corr_image.data.shape[2]
sr_h, sr_w = h * render_factor, w * render_factor

sr_equal_rgb = np.zeros((sr_h, sr_w, 3), dtype=np.float32)
prob_map = np.zeros((sr_h, sr_w), dtype=np.float32)

# 保守版：编号/计数修正很轻，避免让低计数 emitter 被过度放大
emitter_n_locs = np.asarray([em.n_locs for em in emitters], dtype=float)

if hasattr(corr_locs, "lp") and corr_locs.lp is not None:
    emitter_lp = np.asarray([
        np.median(corr_locs.lp[em.loc_idx]) if em.n_locs > 0 else np.nan
        for em in emitters
    ], dtype=float)
else:
    emitter_lp = np.full(n_emitters, np.nan, dtype=float)

if np.any(np.isfinite(emitter_n_locs)):
    n_count = np.clip(emitter_n_locs, 1.0, None)
    emitter_lp_eff = emitter_lp.copy()
    valid = np.isfinite(emitter_lp_eff)
    emitter_lp_eff[valid] = emitter_lp[valid] / np.sqrt(n_count[valid])
else:
    emitter_lp_eff = emitter_lp.copy()

finite_lp = np.isfinite(emitter_lp_eff)
if np.any(finite_lp):
    lp_ref = float(np.nanmedian(emitter_lp_eff[finite_lp]))
    lp_ref = max(lp_ref, 1e-6)
    lp_min = float(np.nanpercentile(emitter_lp_eff[finite_lp], 5))
    lp_max = float(np.nanpercentile(emitter_lp_eff[finite_lp], 95))
    lp_min = max(lp_min, 1e-3)
    lp_max = max(lp_max, lp_min + 1e-3)
else:
    lp_ref = 20.0
    lp_min, lp_max = 0.0, 1.0

# 左图：点状高对比渲染，但用保守有效不确定性给颜色
for i, em in enumerate(emitters):
    if em.n_locs <= 0:
        continue

    x0 = float(np.mean(corr_locs.x[em.loc_idx]))
    y0 = float(np.mean(corr_locs.y[em.loc_idx]))
    lp_nm = float(emitter_lp_eff[i]) if np.isfinite(emitter_lp_eff[i]) else lp_ref

    t = np.clip((lp_nm - lp_min) / max(lp_max - lp_min, 1e-6), 0.0, 1.0)
    color = np.array([
        0.10 + 0.90 * t,
        0.80 * (1.0 - 0.75 * t),
        0.18 + 0.12 * (1.0 - t)
    ], dtype=np.float32)

    x_sr = x0 * render_factor
    y_sr = y0 * render_factor

    radius = 1
    y_min = max(0, int(np.floor(y_sr)) - radius)
    y_max = min(sr_h, int(np.ceil(y_sr)) + radius + 1)
    x_min = max(0, int(np.floor(x_sr)) - radius)
    x_max = min(sr_w, int(np.ceil(x_sr)) + radius + 1)
    yy, xx = np.mgrid[y_min:y_max, x_min:x_max]
    dist2 = (xx - x_sr) ** 2 + (yy - y_sr) ** 2
    patch = np.exp(-dist2 / (2.0 * 0.9 ** 2))
    patch = np.clip(patch * 2.0, 0.0, 1.0)
    sr_equal_rgb[y_min:y_max, x_min:x_max] = np.maximum(
        sr_equal_rgb[y_min:y_max, x_min:x_max],
        patch[..., None] * color[None, None, :]
    )

# 右图：真正的超分辨概率图渲染：把 emitter 当作带权重的 Gaussian 混合分布叠加成图像
# 这是“概率分布视角”的 SR 重建，不是单纯散点扩散；高不确定性和背景点被显著下权重
raw_lp = emitter_lp.copy()
finite_raw = np.isfinite(raw_lp)
if np.any(finite_raw):
    lp_raw_ref = float(np.nanmedian(raw_lp[finite_raw]))
    lp_raw_ref = max(lp_raw_ref, 1e-6)
else:
    lp_raw_ref = 20.0

# 可信度建模：低不确定性 + 足够定位次数 -> 更可信；高不确定性、背景点 -> 强烈压低
emitter_trust = np.zeros(n_emitters, dtype=np.float32)
valid = np.isfinite(emitter_lp_eff)
if np.any(valid):
    lp_eff_valid = emitter_lp_eff[valid]
    count_valid = emitter_n_locs[valid]
    count_ref = float(np.nanmedian(count_valid)) if np.any(np.isfinite(count_valid)) else 1.0
    count_ref = max(count_ref, 1.0)

    if len(lp_eff_valid) > 1:
        lp_cut = float(np.nanpercentile(lp_eff_valid, lp_cut_quantile))
    else:
        lp_cut = float(np.max(lp_eff_valid))

    trust = np.clip(1.0 - (lp_eff_valid - lp_min) / max(lp_max - lp_min, 1e-6), 0.0, 1.0)
    trust *= np.clip(np.sqrt(count_valid / count_ref), 0.5, 2.0)

    # 可调压低强度：
    # downweight_strength = 0.0 -> 不压低
    # downweight_strength = 1.0 -> 中等压低
    # downweight_strength = 2.0 -> 更强压低
    low_confidence_weight = 0.15 / max(downweight_strength, 1e-3)
    trust *= np.where(lp_eff_valid <= lp_cut, 1.0, low_confidence_weight)
    emitter_trust[valid] = trust

if np.any(emitter_trust > 0):
    emitter_trust = np.clip(emitter_trust, 0.0, 1.0)
    emitter_trust = emitter_trust / max(np.nanmax(emitter_trust), 1e-6)

for i, em in enumerate(emitters):
    if em.n_locs <= 0:
        continue
    if emitter_trust[i] <= min_trust:
        continue

    x0 = float(np.mean(corr_locs.x[em.loc_idx]))
    y0 = float(np.mean(corr_locs.y[em.loc_idx]))
    lp_raw = float(raw_lp[i]) if np.isfinite(raw_lp[i]) else lp_raw_ref
    weight = float(emitter_trust[i])

    # gmm_width_scale 可手动调大/调小；如果 lp 偏小导致 Gaussian 过窄，可调到 1.5~2.5
    sigma_img = gmm_width_scale * (0.8 + 1.1 * (lp_raw / max(lp_raw_ref, 1e-6))) * (0.55 + 0.75 * weight)
    sigma_img = np.clip(sigma_img, 0.8, 6.0)
    amp_img = (0.8 + 0.5 * (lp_raw / max(lp_raw_ref, 1e-6))) * weight

    x_sr = x0 * render_factor
    y_sr = y0 * render_factor
    radius_img = int(np.ceil(4.0 * sigma_img))
    y_min = max(0, int(np.floor(y_sr)) - radius_img)
    y_max = min(sr_h, int(np.ceil(y_sr)) + radius_img + 1)
    x_min = max(0, int(np.floor(x_sr)) - radius_img)
    x_max = min(sr_w, int(np.ceil(x_sr)) + radius_img + 1)
    yy, xx = np.mgrid[y_min:y_max, x_min:x_max]
    patch_img = np.exp(-((xx - x_sr) ** 2 + (yy - y_sr) ** 2) / (2.0 * sigma_img ** 2))
    prob_map[y_min:y_max, x_min:x_max] += amp_img * patch_img

if prob_map.max() > 0:
    prob_map /= prob_map.max()
    prob_map = np.clip(prob_map, 0.0, 1.0)
    prob_map = prob_map ** 0.85

if sr_equal_rgb.max() > 0:
    sr_equal_rgb /= sr_equal_rgb.max()
    sr_equal_rgb = np.clip(sr_equal_rgb, 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(17, 7), dpi=180, gridspec_kw={'wspace': 0.35})
for ax in axes:
    ax.set_aspect('equal')
    ax.set_facecolor('black')

ax = axes[0]
ax.imshow(sr_equal_rgb, origin='upper', interpolation='nearest',
          extent=[0, sr_w, sr_h, 0], vmin=0.0, vmax=1.0)
ax.set_title('Equal brightness per emitter\ncolor = effective localization uncertainty')
ax.set_xlabel('x (SR pixel)')
ax.set_ylabel('y (SR pixel)')

cmap_redgreen = LinearSegmentedColormap.from_list(
    'red_green', ['#2ecc71', '#f4d03f', '#e74c3c']
)
sm = plt.cm.ScalarMappable(cmap=cmap_redgreen, norm=plt.Normalize(vmin=lp_min, vmax=lp_max))
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.04)
cb.ax.set_facecolor('black')
cb.ax.set_title('Effective uncertainty', color='white', fontsize=9, pad=8)
cb.set_label('Effective localization uncertainty (px)')

cyan_hot = LinearSegmentedColormap.from_list(
    'cyan_hot', ['#000000', '#005C5C', '#20BDB5', '#DFFFF5']
)


ax = axes[1]
img_lp = np.clip(prob_map, 0.0, None)
vmin, vmax = np.percentile(img_lp[img_lp > 0], (1, 99.5)) if np.any(img_lp > 0) else (0.0, 1.0)
if vmax <= vmin:
    vmin, vmax = 0.0, 1.0
ax.imshow(img_lp, cmap='hot', origin='upper', interpolation='nearest',
          extent=[0, sr_w, sr_h, 0], vmin=vmin, vmax=vmax)
ax.set_title(
    'GMM probability map\n'
    f'width×{gmm_width_scale:g}, downweight={downweight_strength}, '
    f'lp_cut={lp_cut_quantile:g}, min_trust={min_trust:g}, '
    f'trusted={int(np.sum(emitter_trust > trusted_emitters_threshold))}'
)
ax.set_xlabel('x (SR pixel)')
ax.set_ylabel('y (SR pixel)')

sr_img_equal = sr_equal_rgb
sr_img_lp = img_lp

## 评估

In [ ]:
# ===== FRC 分辨率评估（每半独立聚类 + 与最终图一致的 per-emitter 渲染）=====
# 参考 Nieuwenhuizen et al., Nat. Methods 10, 557 (2013)。
#
# 流程（FRC_RENDER='per_emitter' 且 FRC_RECLUSTER=True）：
#   corr_locs --分半--> locs_A / locs_B
#             --各自 analyze_emitters_sm（同一套参数）--> emitters_A / emitters_B
#             --各自按 prob_map 的配方渲染--> img_A / img_B
#             --FRC--> 分辨率
# 每一半的 emitter 集合、簇数量、每簇定位数都是独立得到的，所以两幅图的"颗粒"
# 结构本身就不一样 —— 这正是能体现**结构分辨能力**的地方。
#
# 其它开关：
#   FRC_RECLUSTER=False -> 两半共用全量聚类结果，只在各自存活定位上重算统计量
#   FRC_RENDER='per_loc' -> 逐定位画核（更简单，但与 prob_map 不是同一张图）
#   FRC_SPLIT -> 'loco_random' | 'frame_odd_even' | 'frame_half' | 'emitter_parity' | 'emitter_random'
#                注意：独立聚类下 FRC_SPLIT 只决定"哪些定位进哪一半"，聚类不再共享。
import time

from scipy.ndimage import gaussian_filter

FRC_SPLIT       = 'loco_random'    # 见 SPLIT_DESC
FRC_RENDER      = 'per_emitter'    # 'per_emitter' | 'per_loc'
FRC_RECLUSTER   = True             # True = 每一半独立聚类后再渲染
FRC_SEED        = 0
FRC_SIGMA_SCALE = 1.0              # 仅 per_loc 生效
FRC_USE_HANN    = False
FRC_SMOOTH      = 3

# 与上面 Emitter Analysis 那一格保持一致
FRC_CLUSTER_KW = dict(search_radius=0.6, max_gap=1000, min_frames=3,
                      ecc_p=1e-3, motion='static', do_merge=False)

SPLIT_DESC = {
    'loco_random':    'gold standard：逐条定位随机二分',
    'frame_odd_even': '按帧号奇偶二分（偶数帧 / 奇数帧）',
    'frame_half':     '按采集前半 / 后半二分',
    'emitter_parity': '按 emitter 序号奇偶二分（分子级）',
    'emitter_random': '随机二分 emitter（分子级）',
}


def split_masks(mode, n_loc, seed=0):
    """返回 (mask_a, mask_b)，长度均 = n_loc。"""
    if mode == 'loco_random':
        s = np.random.default_rng(seed).random(n_loc) < 0.5

    elif mode == 'frame_odd_even':
        s = (np.asarray(corr_locs.frame) % 2) == 0

    elif mode == 'frame_half':
        s = np.asarray(corr_locs.frame) < (corr_image.n_frames / 2)

    elif mode in ('emitter_parity', 'emitter_random'):
        le = np.full(n_loc, -1, dtype=np.int64)
        for i, em in enumerate(emitters):
            if em.n_locs > 0:
                le[np.asarray(em.loc_idx)] = i
        if mode == 'emitter_parity':
            g = (np.arange(len(emitters)) % 2) == 0
        else:
            g = np.random.default_rng(seed).random(len(emitters)) < 0.5
        s = np.zeros(n_loc, dtype=bool)
        ok = le >= 0
        s[ok] = g[le[ok]]
        n_orphan = int((~ok).sum())
        if n_orphan:
            s[~ok] = np.random.default_rng(seed + 1).random(n_orphan) < 0.5
            print(f"  注意：{n_orphan} 条定位不属于任何 emitter（全量聚类时未连上），按随机分配")
        if FRC_RECLUSTER:
            print("  FRC_RECLUSTER=True：聚类会在每一半内部重做，此处的归属只用于分半")

    else:
        raise ValueError(f"未知 FRC_SPLIT={mode!r}，可选：{list(SPLIT_DESC)}")

    return s, ~s


def render_emitters_gmm(locs, ems, shape, sel=None):
    """复刻 prob_map 的 per-emitter GMM 配方。

    locs : Localizations（可能是子集，loc_idx 指向它）
    ems  : 与 locs 配套的 emitter 列表
    sel  : 可选布尔掩码（长度 = len(locs)），只保留其中的定位（不重聚类时用）
    """
    H, W = shape
    n_em = len(ems)
    n_e = np.zeros(n_em, dtype=np.float64)
    lp_raw = np.full(n_em, np.nan)
    xc = np.full(n_em, np.nan)
    yc = np.full(n_em, np.nan)

    for i, em in enumerate(ems):
        idx = np.asarray(em.loc_idx)
        if idx.size == 0:
            continue
        if sel is not None:
            idx = idx[sel[idx]]
            if idx.size == 0:
                continue
        n_e[i] = idx.size
        lp_raw[i] = float(np.median(locs.lp[idx]))
        xc[i] = float(np.mean(locs.x[idx]))
        yc[i] = float(np.mean(locs.y[idx]))

    keep = n_e > 0
    n_e, lp_raw, xc, yc = n_e[keep], lp_raw[keep], xc[keep], yc[keep]
    if n_e.size == 0:
        return np.zeros(shape, np.float32), np.zeros(0)

    n_count = np.clip(n_e, 1.0, None)
    lp_eff = lp_raw / np.sqrt(n_count)

    fin = np.isfinite(lp_eff)
    if fin.any():
        lo_l = max(float(np.nanpercentile(lp_eff[fin], 5)), 1e-3)
        hi_l = max(float(np.nanpercentile(lp_eff[fin], 95)), lo_l + 1e-3)
    else:
        lo_l, hi_l = 0.0, 1.0

    trust = np.zeros(n_e.size)
    if fin.any():
        cnt = n_e[fin]
        count_ref_l = max(float(np.nanmedian(cnt)), 1.0)
        lp_cut_l = (float(np.nanpercentile(lp_eff[fin], lp_cut_quantile))
                    if int(fin.sum()) > 1 else float(np.max(lp_eff[fin])))
        low_conf = 0.15 / max(downweight_strength, 1e-3)
        t = np.clip(1.0 - (lp_eff[fin] - lo_l) / max(hi_l - lo_l, 1e-6), 0.0, 1.0)
        t = t * np.clip(np.sqrt(cnt / count_ref_l), 0.5, 2.0)
        t = t * np.where(lp_eff[fin] <= lp_cut_l, 1.0, low_conf)
        trust[fin] = t
    if trust.max() > 0:
        trust = np.clip(trust, 0.0, 1.0) / max(float(trust.max()), 1e-6)

    fin_raw = np.isfinite(lp_raw)
    lp_raw_ref_l = max(float(np.nanmedian(lp_raw[fin_raw])), 1e-6) if fin_raw.any() else 20.0

    img = np.zeros(shape, dtype=np.float32)
    sig_used = []
    for k in range(n_e.size):
        w = float(trust[k])
        if w <= min_trust:
            continue
        lpr = float(lp_raw[k]) if np.isfinite(lp_raw[k]) else lp_raw_ref_l
        sig = gmm_width_scale * (0.8 + 1.1 * (lpr / lp_raw_ref_l)) * (0.55 + 0.75 * w)
        sig = float(np.clip(sig, 0.8, 6.0))
        amp = (0.8 + 0.5 * (lpr / lp_raw_ref_l)) * w

        xs, ys = float(xc[k]) * render_factor, float(yc[k]) * render_factor
        rad = int(np.ceil(4.0 * sig))
        yl = max(0, int(np.floor(ys)) - rad)
        yh = min(H, int(np.ceil(ys)) + rad + 1)
        xl = max(0, int(np.floor(xs)) - rad)
        xh = min(W, int(np.ceil(xs)) + rad + 1)
        if yl >= yh or xl >= xh:
            continue
        yy, xx = np.mgrid[yl:yh, xl:xh]
        img[yl:yh, xl:xh] += amp * np.exp(-((xx - xs) ** 2 + (yy - ys) ** 2) / (2.0 * sig ** 2))
        sig_used.append(sig)

    return img, np.asarray(sig_used)


def render_locs_gmm(xs, ys, shape, sigma):
    """逐定位高斯求和：亚像素双线性投放 + 固定 σ 高斯模糊。"""
    H, W = shape
    xs = np.clip(np.asarray(xs, np.float64), 0.0, W - 1.001)
    ys = np.clip(np.asarray(ys, np.float64), 0.0, H - 1.001)
    x0 = np.floor(xs).astype(np.int64)
    y0 = np.floor(ys).astype(np.int64)
    fx, fy = xs - x0, ys - y0

    acc = np.zeros(H * W, dtype=np.float64)
    for dx, dy, ww in ((0, 0, (1 - fx) * (1 - fy)),
                       (1, 0, fx * (1 - fy)),
                       (0, 1, (1 - fx) * fy),
                       (1, 1, fx * fy)):
        ix = np.clip(x0 + dx, 0, W - 1)
        iy = np.clip(y0 + dy, 0, H - 1)
        acc += np.bincount(iy * W + ix, weights=ww, minlength=H * W)

    img = acc.reshape(H, W).astype(np.float32)
    if sigma > 0:
        img = gaussian_filter(img, sigma, mode='constant', truncate=4.0)
    return img


def frc_curve(im1, im2, n_bins=None, hann=False):
    """返回 (径向频率中心, FRC)。频率单位 cycles / SR 像素，最大值 0.5。"""
    if hann:
        wy = np.hanning(im1.shape[0])[:, None]
        wx = np.hanning(im1.shape[1])[None, :]
        im1, im2 = im1 * wy * wx, im2 * wy * wx

    F1 = np.fft.fftshift(np.fft.fft2(im1))
    F2 = np.fft.fftshift(np.fft.fft2(im2))
    num = (F1 * np.conj(F2)).real
    p1, p2 = np.abs(F1) ** 2, np.abs(F2) ** 2

    h, w = im1.shape
    fy = np.fft.fftshift(np.fft.fftfreq(h))[:, None]
    fx = np.fft.fftshift(np.fft.fftfreq(w))[None, :]
    fr = np.sqrt(fx ** 2 + fy ** 2)

    if n_bins is None:
        n_bins = max(8, min(h, w) // 2)
    edges = np.linspace(0.0, 0.5, n_bins + 1)
    which = np.clip(np.digitize(fr.ravel(), edges) - 1, 0, n_bins - 1)

    fc = 0.5 * (edges[:-1] + edges[1:])
    frc = np.full(n_bins, np.nan)
    nr, p1r, p2r = num.ravel(), p1.ravel(), p2.ravel()
    for k in range(n_bins):
        m = which == k
        if not m.any():
            continue
        den = np.sqrt(p1r[m].sum() * p2r[m].sum())
        if den > 0:
            frc[k] = nr[m].sum() / den
    return fc, frc


def frc_resolution(fc, frc, thresh=1.0 / 7.0, smooth=1):
    """FRC 首次跌破阈值处的频率（线性插值）。smooth>1 时先滑动平均。"""
    ok = np.isfinite(frc)
    fc, frc = fc[ok], frc[ok]
    if smooth and smooth > 1 and frc.size >= smooth:
        k = np.ones(smooth) / smooth
        frc = np.convolve(frc, k, mode='valid')
        fc = np.convolve(fc, k, mode='valid')
    below = np.where(frc < thresh)[0]
    if below.size == 0:
        return np.nan
    k = int(below[0])
    if k == 0:
        return float(fc[0])
    v0, v1 = frc[k - 1], frc[k]
    f0, f1 = fc[k - 1], fc[k]
    return float(f0) if v0 == v1 else float(f0 + (v0 - thresh) * (f1 - f0) / (v0 - v1))


# ---- 分半 ----
if FRC_SPLIT not in SPLIT_DESC:
    raise ValueError(f"FRC_SPLIT 必须是 {list(SPLIT_DESC)} 之一")

_h_f, _w_f = int(corr_image.data.shape[1]), int(corr_image.data.shape[2])
sr_h, sr_w = _h_f * render_factor, _w_f * render_factor
px_sr_nm = float(pixel_size) / float(render_factor)
n_loc = len(corr_locs)
_n_frames = int(corr_image.n_frames)

print(f"分半方式        : {FRC_SPLIT} — {SPLIT_DESC[FRC_SPLIT]}")
print(f"渲染方式        : {FRC_RENDER} | 独立聚类: {FRC_RECLUSTER}")

_sel_a, _sel_b = split_masks(FRC_SPLIT, n_loc, seed=FRC_SEED)
print(f"定位总数        : {n_loc}  ->  A {int(_sel_a.sum())} / B {int(_sel_b.sum())}")
print(f"SR 网格         : {sr_w} x {sr_h} px, {px_sr_nm:.2f} nm/px (render_factor={render_factor})")

sig_sr = np.nan
_ems_a = _ems_b = None       # 只有独立聚类分支才会填
if FRC_RENDER == 'per_emitter':
    if FRC_RECLUSTER:
        _locs_a, _locs_b = corr_locs[_sel_a], corr_locs[_sel_b]
        _t0 = time.time()
        print("每一半独立聚类中 ...")
        _ems_a, _ = analyze_emitters_sm(_locs_a, n_frames=_n_frames,
                                     first_frame_range=(0, _n_frames - 1),
                                     **FRC_CLUSTER_KW)
        _ems_b, _ = analyze_emitters_sm(_locs_b, n_frames=_n_frames,
                                     first_frame_range=(0, _n_frames - 1),
                                     **FRC_CLUSTER_KW)
        print(f"独立聚类完成：A {len(_ems_a)} / B {len(_ems_b)} 个 emitter "
              f"(全量聚类为 {len(emitters)})，用时 {time.time() - _t0:.0f}s")
        _n_in_a = sum(int(em.n_locs) for em in _ems_a)
        _n_in_b = sum(int(em.n_locs) for em in _ems_b)
        print(f"  进入渲染的定位: A {_n_in_a} / {len(_locs_a)}  "
              f"B {_n_in_b} / {len(_locs_b)}  （未通过聚类/过滤的被丢弃）")
        img_a, sig_a = render_emitters_gmm(_locs_a, _ems_a, (sr_h, sr_w))
        img_b, sig_b = render_emitters_gmm(_locs_b, _ems_b, (sr_h, sr_w))
    else:
        img_a, sig_a = render_emitters_gmm(corr_locs, emitters, (sr_h, sr_w), sel=_sel_a)
        img_b, sig_b = render_emitters_gmm(corr_locs, emitters, (sr_h, sr_w), sel=_sel_b)
        print(f"实际渲染 emitter: A {sig_a.size} / B {sig_b.size} (全量 {len(emitters)}，共用聚类)")

    _all_sig = np.concatenate([sig_a, sig_b]) if sig_a.size + sig_b.size else np.zeros(1)
    print(f"核宽 sigma      : 中位 {np.median(_all_sig):.2f} SR px = "
          f"{np.median(_all_sig) * px_sr_nm:.1f} nm，范围 {_all_sig.min():.2f} ~ {_all_sig.max():.2f} SR px")
else:
    sig_sr = FRC_SIGMA_SCALE * float(np.nanmedian(corr_locs.lp)) * render_factor
    img_a = render_locs_gmm(corr_locs.x[_sel_a] * render_factor,
                            corr_locs.y[_sel_a] * render_factor, (sr_h, sr_w), sig_sr)
    img_b = render_locs_gmm(corr_locs.x[_sel_b] * render_factor,
                            corr_locs.y[_sel_b] * render_factor, (sr_h, sr_w), sig_sr)
    sig_a = sig_b = np.zeros(0)          # per_loc 没有 per-emitter 核宽
    _all_sig = np.array([sig_sr])
    print(f"渲染高斯 sigma  : {sig_sr:.2f} SR px = {sig_sr * px_sr_nm:.1f} nm "
          f"(= {FRC_SIGMA_SCALE:g} x median(lp)={float(np.nanmedian(corr_locs.lp)):.3f} px)")

# ---- FRC ----
fc, frc = frc_curve(img_a, img_b, hann=FRC_USE_HANN)
f_raw = frc_resolution(fc, frc, smooth=1)
f_cut = frc_resolution(fc, frc, smooth=FRC_SMOOTH)

if np.isfinite(f_cut) and f_cut > 0:
    res_nm = px_sr_nm / f_cut
    print(f"FRC(1/7) 交点   : {f_cut:.4f} cycles/SR-px "
          f"({1000.0 * f_cut / px_sr_nm:.2f} 1/µm) -> {1 / f_cut:.2f} SR px = {res_nm:.1f} nm")
    if np.isfinite(f_raw):
        print(f"  (未平滑的单 bin 交点 {f_raw:.4f} -> {px_sr_nm / f_raw:.1f} nm)")
else:
    res_nm = np.nan
    print("FRC 全频段都在 1/7 以上，未跌破阈值")
print(f"Nyquist 极限    : 0.5 cycles/SR-px = {2 * px_sr_nm:.1f} nm")

_caveat = {
    'loco_random':    f"逐定位随机分半 + 独立聚类：两半共享分子但聚类各自重做，"
                      f"emitter 集合、簇长、每簇定位数都不同",
    'frame_odd_even': "按帧号奇偶分半 + 独立聚类；时间上独立，若存在漂白/漂移两半信噪会不同",
    'frame_half':     "按采集前后半段分半 + 独立聚类；漂白会让后半段定位数偏少",
    'emitter_parity': "分子级分半（基于全量聚类）+ 独立聚类；两半分子集合不相交",
    'emitter_random': "分子级随机分半（基于全量聚类）+ 独立聚类；两半分子集合不相交",
}[FRC_SPLIT]
print(f"解读            : {_caveat}")
print("提示            : 换 FRC_SEED 可看随机类模式的波动；FRC_USE_HANN=True 可压边缘伪影。")

# ---- 出图 ----
f_um = fc * 1000.0 / px_sr_nm          # cycles / µm
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), dpi=150)

for ax, im, ttl in [(axes[0], img_a, f'half A (locs={int(_sel_a.sum())})'),
                    (axes[1], img_b, f'half B (locs={int(_sel_b.sum())})')]:
    vmax = np.percentile(im[im > 0], 99.5) if np.any(im > 0) else 1.0
    ax.imshow(im, cmap='hot', origin='upper', interpolation='nearest',
              vmin=0, vmax=max(vmax, 1e-6))
    ax.set_title(ttl, fontsize=10)
    ax.set_xlabel('x (SR pixel)')
    ax.set_ylabel('y (SR pixel)')
    ax.set_aspect('equal')

ax = axes[2]
ax.plot(f_um, frc, lw=1.5, label='FRC')
ax.axhline(1.0 / 7.0, color='r', ls='--', lw=1.0, label='1/7 threshold')
if np.isfinite(f_cut):
    ax.axvline(f_cut * 1000.0 / px_sr_nm, color='g', ls=':', lw=1.2, label=f'{res_nm:.0f} nm')
ax.set_xlabel('spatial frequency (1/µm)')
ax.set_ylabel('FRC')
ax.set_ylim(-0.2, 1.05)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc='upper right')
ax.set_title(f'FRC — {FRC_RENDER}, split={FRC_SPLIT}, recluster={FRC_RECLUSTER}', fontsize=10)

fig.tight_layout()
plt.show()

## 导出环节

In [ ]:
# ===== 把右图（GMM 概率图）保存为高清 TIFF =====
import os
import numpy as np
import tifffile

out_dir = os.path.dirname(src)
tag = os.path.splitext(os.path.basename(src))[0]

# SR 图的物理像素尺寸 = 原像素 / render_factor
px_sr_nm = float(pixel_size) / float(render_factor)
res = 1.0 / (px_sr_nm / 1000.0)          # pixels per µm（与 smlm.io.save_tif 的约定一致）
ij_meta = {'axes': 'YX', 'unit': 'um'}

# 1) float32：保留完整动态范围、零量化损失（做定量/再渲染请用这个）
sr_f32 = np.asarray(prob_map, dtype=np.float32)
p32 = os.path.join(out_dir, f"{tag}_SR_gmm_float32.tif")
tifffile.imwrite(p32, sr_f32, imagej=True, metadata=ij_meta,
                 resolution=(res, res), compression='zlib')

# 2) uint16：按显示用的 1–99.5% 分位拉伸，ImageJ / Fiji 直接可看
img = np.clip(sr_f32, 0.0, None)
if np.any(img > 0):
    lo, hi = np.percentile(img[img > 0], (1, 99.5))
else:
    lo, hi = 0.0, 1.0
if hi <= lo:
    lo, hi = 0.0, 1.0
sr_u16 = (np.clip((img - lo) / (hi - lo), 0.0, 1.0) * 65535.0 + 0.5).astype(np.uint16)
p16 = os.path.join(out_dir, f"{tag}_SR_gmm_16bit.tif")
tifffile.imwrite(p16, sr_u16, imagej=True, metadata=ij_meta,
                 resolution=(res, res), compression='zlib')

print(f"SR 网格   : {sr_f32.shape[1]} x {sr_f32.shape[0]} px, "
      f"{px_sr_nm:.2f} nm/px (render_factor={render_factor})")
print(f"float32  : {p32}   range {sr_f32.min():.4g} .. {sr_f32.max():.4g}")
print(f"uint16   : {p16}   stretch {lo:.4g} .. {hi:.4g} -> 0..65535")

In [ ]:
# ===== 导出：漂移校正后的 tif + ThunderSTORM 格式定位（存到 <tag>_FRC/）=====
# ThunderSTORM CSV 列（与 ThunderSTORM 自己导出的 2D 表一致）：
#   id, frame, x [nm], y [nm], sigma [nm], intensity [photon],
#   offset [photon], bkgstd [photon], uncertainty_xy [nm]
# 映射关系：
#   frame   = corr_locs.frame + TS_FRAME_BASE          # 0-based -> ThunderSTORM 1-based
#   x/y     = corr_locs.x/y * pixel_size              # 校正后图像的坐标系（已裁切）
#   sigma   = corr_locs.sigma * pixel_size            # PSF 高斯 sigma（px -> nm）
#   intensity = corr_locs.photons
#   offset  = corr_locs.bg                            # 每像素背景光子数
#   bkgstd  = sqrt(offset)                            # 散粒噪声估计（无逐点实测值）
#   uncertainty_xy = corr_locs.lp * pixel_size        # 侧向定位精度 px -> nm
import json
import os

import numpy as np
import pandas as pd
import tifffile

SAVE_CORR_TIF  = True
SAVE_TS_LOCS   = True
CORR_TIF_DTYPE = 'uint16'     # 'uint16'（体积小、够用）| 'float32'（无损）
TS_FRAME_BASE  = 1            # 第一帧写成的帧号（ThunderSTORM 约定从 1 开始；要 0 起就改 0）

_tag = os.path.splitext(os.path.basename(src))[0]
_out = os.path.join(os.path.dirname(src), f"{_tag}_FRC")
os.makedirs(_out, exist_ok=True)

_res = 1.0 / (float(pixel_size) / 1000.0)          # pixels per µm
_img = corr_image.data
_nf, _hh, _ww = int(_img.shape[0]), int(_img.shape[1]), int(_img.shape[2])
print("输出目录:", _out)
print(f"校正后图像: {_nf} x {_hh} x {_ww}, dtype={_img.dtype}, "
      f"值域 {float(np.min(_img)):.2f} ~ {float(np.max(_img)):.2f}")
print(f"定位       : {len(corr_locs)} 条, 帧 {int(corr_locs.frame.min())} .. "
      f"{int(corr_locs.frame.max())}")

# ---- 1) 校正后的 tif ----
if SAVE_CORR_TIF:
    p_tif = os.path.join(_out, f"{_tag}_drift_corrected.tif")
    if CORR_TIF_DTYPE == 'uint16':
        _arr = np.empty((_nf, _hh, _ww), dtype=np.uint16)
        for _i in range(_nf):                       # 逐帧转换，避免整幅 float64 中间量
            _arr[_i] = np.clip(np.rint(_img[_i]), 0, 65535)
    else:
        _arr = np.asarray(_img, dtype=np.float32)
    tifffile.imwrite(p_tif, _arr, imagej=True,
                     metadata={'axes': 'TYX', 'unit': 'um'},
                     resolution=(_res, _res), compression='zlib')
    print(f"saved tif : {p_tif}  ({os.path.getsize(p_tif) / 1e9:.2f} GB, {CORR_TIF_DTYPE})")
    del _arr

# ---- 2) ThunderSTORM CSV ----
if SAVE_TS_LOCS:
    p_csv = os.path.join(_out, f"{_tag}_locs_thunderstorm.csv")
    _n = len(corr_locs)
    _bg = np.clip(np.asarray(corr_locs.bg, np.float64), 0.0, None)
    _df = pd.DataFrame({
        'id': np.arange(1, _n + 1, dtype=np.int64),
        'frame': np.asarray(corr_locs.frame, np.int64) + TS_FRAME_BASE,
        'x [nm]': np.asarray(corr_locs.x, np.float64) * float(pixel_size),
        'y [nm]': np.asarray(corr_locs.y, np.float64) * float(pixel_size),
        'sigma [nm]': np.asarray(corr_locs.sigma, np.float64) * float(pixel_size),
        'intensity [photon]': np.asarray(corr_locs.photons, np.float64),
        'offset [photon]': _bg,
        'bkgstd [photon]': np.sqrt(_bg),
        'uncertainty_xy [nm]': np.asarray(corr_locs.lp, np.float64) * float(pixel_size),
    })
    _hdr = ('"id","frame","x [nm]","y [nm]","sigma [nm]","intensity [photon]",'
            '"offset [photon]","bkgstd [photon]","uncertainty_xy [nm]"')
    with open(p_csv, 'w', newline='', encoding='utf-8') as fh:
        fh.write(_hdr + '\n')
        _df.to_csv(fh, index=False, header=False, float_format='%.3f')
    print(f"saved locs: {p_csv}  ({os.path.getsize(p_csv) / 1e6:.1f} MB, {_n} 行)")

    _info = dict(
        format='ThunderSTORM 2D CSV',
        source=os.path.basename(str(src)),
        corrected_tif=(f"{_tag}_drift_corrected.tif" if SAVE_CORR_TIF else None),
        n_locs=int(_n),
        pixel_size_nm=float(pixel_size),
        frame_base=int(TS_FRAME_BASE),
        coord_frame=f'drift-corrected, cropped {_hh}x{_ww} px',
        columns=dict(
            id='1..N',
            frame='corr_locs.frame + TS_FRAME_BASE（第一帧 = frame_base）',
            **{'x [nm]': 'corr_locs.x * pixel_size',
               'y [nm]': 'corr_locs.y * pixel_size',
               'sigma [nm]': 'corr_locs.sigma(px) * pixel_size —— DECODE 训练用的高斯 PSF sigma',
               'intensity [photon]': 'corr_locs.photons',
               'offset [photon]': 'corr_locs.bg（每像素背景光子数）',
               'bkgstd [photon]': 'sqrt(offset)，散粒噪声估计，非逐点实测',
               'uncertainty_xy [nm]': 'corr_locs.lp(px) * pixel_size'},
        ),
        saved=time.strftime('%Y-%m-%dT%H:%M:%S'),
    )
    p_info = os.path.join(_out, f"{_tag}_locs_thunderstorm.json")
    with open(p_info, 'w', encoding='utf-8') as fh:
        json.dump(_info, fh, indent=2, ensure_ascii=False)
    print(f"saved info: {p_info}")


In [ ]:
# ===== 保存 FRC 结果到 raw 文件旁边的 <tag>_FRC/ 目录 =====
# 保存内容（文件名带上 split / render / recluster 等配置，不会互相覆盖）：
#   <name>.png            三联图（两半 SR 图 + FRC 曲线）
#   <name>_curve.csv      频率-FRC 曲线（freq 单位 1/µm 与 cycles/SR-px 各一列）
#   <name>_curve.npz      同上，numpy 格式
#   <name>_halfA.tif      两半的 SR 图，float32（带 µm 分辨率标签）
#   <name>_halfB.tif
#   <name>.json           本次全部参数与结果
#   FRC_summary.csv       所有配置的结果汇总（追加写，便于横向比较）
import csv
import json
import os

import tifffile

SAVE_FRC = True

_tag = os.path.splitext(os.path.basename(src))[0]
_frc_dir = os.path.join(os.path.dirname(src), f"{_tag}_FRC")
_name = f"FRC_{FRC_RENDER}_{FRC_SPLIT}"
if FRC_RENDER == 'per_emitter' and FRC_RECLUSTER:
    _name += "_recluster"
if FRC_RENDER == 'per_loc' and FRC_SIGMA_SCALE != 1.0:
    _name += f"_sig{FRC_SIGMA_SCALE:g}"

print("输出目录:", _frc_dir)
if not SAVE_FRC:
    print("SAVE_FRC=False，未写出")
else:
    os.makedirs(_frc_dir, exist_ok=True)
    _res = 1.0 / (px_sr_nm / 1000.0)          # pixels per µm
    _ij = {'axes': 'YX', 'unit': 'um'}

    # 1) 三联图
    p_png = os.path.join(_frc_dir, f"{_name}.png")
    fig.savefig(p_png, dpi=200, bbox_inches='tight')

    # 2) 曲线
    p_csv = os.path.join(_frc_dir, f"{_name}_curve.csv")
    with open(p_csv, 'w', newline='', encoding='utf-8') as fh:
        w = csv.writer(fh)
        w.writerow(['freq_per_um', 'freq_cycles_per_sr_px', 'FRC'])
        for _f_um, _f_c, _v in zip(f_um, fc, frc):
            w.writerow([f"{_f_um:.6g}", f"{_f_c:.6g}", "" if not np.isfinite(_v) else f"{_v:.6f}"])
    p_npz = os.path.join(_frc_dir, f"{_name}_curve.npz")
    np.savez_compressed(p_npz, f_per_um=f_um, f_cycles_per_sr_px=fc, frc=frc)

    # 3) 两半 SR 图（float32，保留动态范围）
    p_a = os.path.join(_frc_dir, f"{_name}_halfA.tif")
    tifffile.imwrite(p_a, np.asarray(img_a, np.float32), imagej=True, metadata=_ij,
                     resolution=(_res, _res), compression='zlib')
    p_b = os.path.join(_frc_dir, f"{_name}_halfB.tif")
    tifffile.imwrite(p_b, np.asarray(img_b, np.float32), imagej=True, metadata=_ij,
                     resolution=(_res, _res), compression='zlib')

    # 4) 参数与结果摘要
    _ems_a = globals().get('_ems_a')
    _ems_b = globals().get('_ems_b')
    _all_sig = globals().get('_all_sig', np.array([sig_sr]))
    _summary = dict(
        raw_tif=str(src),
        saved=time.strftime('%Y-%m-%dT%H:%M:%S'),
        split=FRC_SPLIT,
        render=FRC_RENDER,
        recluster=bool(FRC_RENDER == 'per_emitter' and FRC_RECLUSTER),
        seed=int(FRC_SEED),
        sigma_scale=float(FRC_SIGMA_SCALE),
        hann=bool(FRC_USE_HANN),
        smooth=int(FRC_SMOOTH),
        cluster_kw=(FRC_CLUSTER_KW if FRC_RENDER == 'per_emitter' else None),
        n_loc_total=int(n_loc),
        n_loc_A=int(_sel_a.sum()),
        n_loc_B=int(_sel_b.sum()),
        n_emitters_full=int(len(emitters)),
        n_emitters_A=(len(_ems_a) if _ems_a is not None else None),
        n_emitters_B=(len(_ems_b) if _ems_b is not None else None),
        n_emitters_rendered_A=int(np.size(sig_a)),
        n_emitters_rendered_B=int(np.size(sig_b)),
        sigma_median_sr_px=float(np.median(_all_sig)),
        sigma_median_nm=float(np.median(_all_sig) * px_sr_nm),
        render_factor=int(render_factor),
        sr_grid_px=[int(sr_w), int(sr_h)],
        px_sr_nm=float(px_sr_nm),
        pixel_size_nm=float(pixel_size),
        f_cut_cycles_per_sr_px=float(f_cut),
        f_cut_per_um=float(1000.0 * f_cut / px_sr_nm),
        resolution_nm=float(res_nm),
        nyquist_nm=float(2 * px_sr_nm),
        files=dict(png=os.path.basename(p_png), curve_csv=os.path.basename(p_csv),
                   curve_npz=os.path.basename(p_npz), halfA=os.path.basename(p_a),
                   halfB=os.path.basename(p_b)),
    )
    p_json = os.path.join(_frc_dir, f"{_name}.json")
    with open(p_json, 'w', encoding='utf-8') as fh:
        json.dump(_summary, fh, indent=2, ensure_ascii=False)

    # 5) 汇总表（追加）
    p_sum = os.path.join(_frc_dir, 'FRC_summary.csv')
    _row = dict(file=_name, split=FRC_SPLIT, render=FRC_RENDER,
                recluster=_summary['recluster'], seed=int(FRC_SEED),
                n_loc_A=_summary['n_loc_A'], n_loc_B=_summary['n_loc_B'],
                n_emitters_A=_summary['n_emitters_A'], n_emitters_B=_summary['n_emitters_B'],
                f_cut_per_um=_summary['f_cut_per_um'],
                resolution_nm=_summary['resolution_nm'], nyquist_nm=_summary['nyquist_nm'])
    _new = not os.path.exists(p_sum)
    with open(p_sum, 'a', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=list(_row))
        if _new:
            w.writeheader()
        w.writerow(_row)

    print("saved:")
    for _p in (p_png, p_csv, p_npz, p_a, p_b, p_json, p_sum):
        print(f"  {os.path.basename(_p):<48} {os.path.getsize(_p) / 1e3:8.1f} kB")
    print(f"分辨率 {res_nm:.1f} nm @ FRC=1/7  (Nyquist {2 * px_sr_nm:.1f} nm)")